# 🚁 UAV Detection - Google Colab Training & Inference

This notebook trains YOLOv11 on Google Colab using `Dataset_Cleaned.zip`.
It saves results to Google Drive and includes an inference demo.

### **Instructions**
1. **Upload**: Upload `Dataset_Cleaned.zip` to your Google Drive in the folder `Computer_Vision`.
2. **Mount Drive**: Run the first cell to connect to Google Drive.
3. **Run**: Run all cells below.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define Drive paths
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Computer_Vision"
ZIP_PATH = f"{DRIVE_PROJECT_PATH}/Dataset_Cleaned.zip"
RESULTS_PATH = f"{DRIVE_PROJECT_PATH}/runs"

In [ ]:
# 2. Install Ultralytics
%pip install ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# 3. Unzip Dataset from Drive
import os
import shutil

extract_path = "/content/Dataset"

# Check if zip is in Drive, otherwise ask to upload locally
if os.path.exists(ZIP_PATH):
    print(f"Found zip in Drive: {ZIP_PATH}")
    source_zip = ZIP_PATH
elif os.path.exists("/content/Dataset_Cleaned.zip"):
    print("Found zip uploaded locally to Colab.")
    source_zip = "/content/Dataset_Cleaned.zip"
else:
    raise FileNotFoundError(f"Count not find {ZIP_PATH}. Please make sure Dataset_Cleaned.zip is in 'Computer_Vision' folder in your Drive.")

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

print("Unzipping...")
!unzip -q "{source_zip}" -d "{extract_path}"
print("Unzip complete.")

In [ ]:
# 4. Configure data.yaml (Auto-Detect Path)
import yaml

def find_dataset_root(start_path):
    """Finds the folder containing 'train/images'."""
    for root, dirs, files in os.walk(start_path):
        if 'train' in dirs and os.path.exists(os.path.join(root, 'train', 'images')):
            return root
    return start_path

dataset_root = find_dataset_root("/content/Dataset")
print(f"Dataset root found at: {dataset_root}")

data_config = {
    'path': dataset_root, 
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['UAV']
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("data.yaml configured.")

In [ ]:
# 5. Train Model
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

print("Starting training...")
results = model.train(
    data="data.yaml",
    epochs=15,
    imgsz=640,
    batch=48,        # Adjusted to 48 (64 caused OOM)
    project="/content/runs",
    name="uav_colab_drive",
    exist_ok=True
)

In [ ]:
# 6. Copy Results to Drive
import os
import shutil

print(f"Copying results to Drive: {RESULTS_PATH}...")
if not os.path.exists(DRIVE_PROJECT_PATH):
    os.makedirs(DRIVE_PROJECT_PATH)

if os.path.exists(RESULTS_PATH):
    shutil.rmtree(RESULTS_PATH)

shutil.copytree("/content/runs", RESULTS_PATH)
print("✅ Saved runs folders to Google Drive!")

In [ ]:
# 7. Display Evaluation Plots (Confusion Matrix & Training Graphs)
from IPython.display import Image, display
import os

# Path to the training run
run_path = "/content/runs/uav_colab_drive"

print("--- Training Results (Loss, Accuracy) ---")
display(Image(filename=f"{run_path}/results.png"))

print("\n--- Confusion Matrix ---")
if os.path.exists(f"{run_path}/confusion_matrix.png"):
    display(Image(filename=f"{run_path}/confusion_matrix.png"))
else:
    print("Confusion matrix not found (may need more epochs or validation data).")

print("\n--- Precision-Recall Curve ---")
if os.path.exists(f"{run_path}/PR_curve.png"):
    display(Image(filename=f"{run_path}/PR_curve.png"))

In [ ]:
# 8. Inference Demo (Test on Validation Image)
import glob

# Find a test image from validation set
val_images = glob.glob(f"{dataset_root}/valid/images/*.jpg")

if val_images:
    test_image = val_images[0] # Pick first one
    print(f"Running inference on: {test_image}")
    
    # Run inference
    model = YOLO(f"{run_path}/weights/best.pt") # Load best trained model
    results = model.predict(test_image, save=True, conf=0.5)
    
    # Display result
    for result in results:
        # Result saved usually in runs/detect/predict
        saved_dir = result.save_dir
        saved_img = f"{saved_dir}/{os.path.basename(test_image)}"
        display(Image(filename=saved_img))
else:
    print("No validation images found to test.")

In [ ]:
# 9. Run on Full Test Set & generate Visuals
print("Running prediction on Test Set...")

# Load best model
model = YOLO(f"{run_path}/weights/best.pt")

# Run prediction on all test images
results = model.predict(
    source=f"{dataset_root}/test/images",
    conf=0.25,
    save=True,
    project="/content/runs/detect",
    name="test_predictions",
    exist_ok=True
)

print("✅ Predictions finished! Visuals saved to /content/runs/detect/test_predictions")

In [ ]:
# 10. Download Everything (Weights + Plots + Test Visuals)
import shutil
from google.colab import files

# 1. Save to Drive (Updates the folder with new test results)
drive_save_path = f"{DRIVE_PROJECT_PATH}/runs_final"
print(f"Copying all results to Drive folder: {drive_save_path}...")
shutil.copytree("/content/runs", drive_save_path, dirs_exist_ok=True)
print("✅ Saved to Drive!")

# 2. Create Zip for Local Download
print("Zipping for download...")
!zip -r /content/final_uav_results.zip /content/runs

# 3. Trigger Download
print("Downloading zip...")
files.download("/content/final_uav_results.zip")